In [15]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, ctx, callback
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#CRUD Python module file name and class name
from animalShelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
#username and password and CRUD Python module name

username = "aacuser"
password = "SNHU123"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)

## Debug
print(len(df.to_dict(orient='records')))
print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()


#Dashboard layout and View
app.layout = html.Div([
    html.Center(html.B(html.H1('Project Two: Andrew Lester'))),
    html.Hr(),
    html.Center(html.Img(src=f'data:image/png;base64,{encoded_image}', style={'height': '150px'})),
    #Filtering options using Radio buttons for required rescue services
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Wilderness Rescue', 'value': 'Wild'},
            {'label': 'Disaster Rescue', 'value': 'Disaster'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='reset',
        labelStyle={'display': 'inline-block'}
#Failed attempts at using standard Buttons 
    #html.Div(className='buttonRow',
             #style={'display' : 'flex'},
             #children=[
                 #html.Button(id='button1', n_clicks=0, children='Water'),
                 #html.Button(id='button2', n_clicks=0, children='Wilderness'),
                 #html.Button(id='button3', n_clicks=0, children='Disaster'),
                 #html.Button(id='button4', n_clicks=0, children='Reset')
             #]
             #dcc.Button('Water Rescue', id='btn-nclicks-1', n_clicks=0),
             #dcc.Button('Wilderness Rescue', id='btn-nclicks-2', n_clicks=0),
             #dcc.Button('Disaster Rescue', id='btn-nclicks-3', n_clicks=0),
             #dcc.Button('Reset', id='btn-nclicks-4', n_clicks=0),
             #html.Div(id='container-button-timestamp')
    ),
    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         #DataTable Styling, Filtering, Selecting, and Paging components
                         style_cell={'textAlign': 'right'},
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable="single",
                         row_selectable="single",
                         row_deletable=False,
                         selected_columns=[],
                         selected_rows=[],
                         page_action="native",
                         page_current= 0,
                         page_size= 10,

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



#Filtering dashboard callback
@app.callback([Output('datatable-id', 'data'),
               Output('datatable-id', 'columns')],
              [Input('filter-type', 'value')]
             )
#failed callbacks for the button attempts
#@app.callback(Output('datatable-id','data'),
              #[Input('button1', 'n_clicks'),
               #Input('button2', 'n_clicks'),
               #Input('button3', 'n_clicks'),
               #Input('button4', 'n_clicks')
              #])
#@app.callback(Output('container-button-timestamp', 'children'),
              #Input('btn-nclicks-1', 'n_clicks'),
              #Input('btn-nclicks-2', 'n_clicks'),
              #Input('btn-nclicks-3', 'n_clicks'),
              #Input('btn-nclicks-4', 'n_clicks')
              #)
    
def update_dashboard(filterType):
    query = {}
    if filterType == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filterType == 'Wild':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filterType == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
        
    df_filter = pd.DataFrame(list(db.read(query)))
    if '_id' in df_filter.columns:
        df_filter.drop(columns=['_id'], inplace=True)
    
    return df_filter.to_dict('records'), [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df_filter.columns]


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    dff = pd.DataFrame.from_dict(viewData)
    
    return [
        dcc.Graph(
            figure = px.pie(dff, names='breed', title='Preferred Animals')
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

def update_map(viewData, index):  
    #If there is no data the map defaults to the center of Austin Texas
    if not viewData:
        return [dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[dl.TileLayer()])]
    #elif index is None:
        #return
    
    dff = pd.DataFrame.from_dict(viewData)
    #Default status
    if selected_rows == []:
        selected_rows = [0]
        
    #Makes sure selected row is within Dataframe
    if row >= len(dff):
        row - 0
        
    #Fetch selected location
    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']
    
    #fetch selected animal name and breed
    animalName = dff.iloc[row].get('name', 'No Name')
    breed = dff.iloc[row].get('breed', 'No Breed')
    # Because we only allow single row selection, the list can be converted to a row index here
    #if index is None:
        #row == 0
    #else: 
        #row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[latitude, longitude], zoom=10, children=[
            dl.TileLayer(),
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(animalName)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(debug=False, port=30641) 

Connection successful
50001
Index(['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed',
       'color', 'date_of_birth', 'datetime', 'monthyear', 'name',
       'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat',
       'location_long', 'age_upon_outcome_in_weeks'],
      dtype='object')


 * Running on http://127.0.0.1:30641/ (Press CTRL+C to quit)
127.0.0.1 - - [18/Apr/2026 05:13:45] "GET /_alive_3f10a5a9-781e-4a2d-ba0d-744aa44f9c51 HTTP/1.1" 200 -


Dash app running on https://concertsaddle-cipherabsorb-3000.codio.io/proxy/30641/


127.0.0.1 - - [18/Apr/2026 05:13:48] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:48] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:50] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:50] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [18/Apr/2026 05:13:50] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [18/Apr/2026 05:13:50] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:50] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:51] "POST /_dash-update-component HTTP/1.1" 500 -
127.0.0.1 - - [18/Apr/2026 05:13:55] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:59] "POST /_dash-update-component HTTP/1.1" 500 -
127.0.0.1 - - [18/Apr/2026 05:13:59] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [18/Apr/2026 05:13:59] "GET /_dash-component-sui